# Hosting MCP Server on Amazon Bedrock AgentCore Runtime - AWS IAM Inbound Authentication

# 在 Amazon Bedrock AgentCore Runtime 上托管 MCP 服务器 - AWS IAM 入站认证

## Overview

## 概述

In this tutorial we will learn how to host MCP (Model Context Protocol) servers on Amazon Bedrock AgentCore Runtime. We will use the Amazon Bedrock AgentCore Python SDK to wrap MCP tools as an MCP server compatible with Amazon Bedrock AgentCore.

在本教程中，我们将学习如何在 Amazon Bedrock AgentCore Runtime 上托管 MCP（模型上下文协议）服务器。我们将使用 Amazon Bedrock AgentCore Python SDK 将 MCP 工具封装为与 Amazon Bedrock AgentCore 兼容的 MCP 服务器。

The Amazon Bedrock AgentCore Python SDK handles the MCP server implementation details so you can focus on your tools' core functionality. It transforms your code into the AgentCore standardized MCP protocol contracts for direct communication.

Amazon Bedrock AgentCore Python SDK 处理 MCP 服务器的实现细节，使您可以专注于工具的核心功能。它将您的代码转换为 AgentCore 标准化的 MCP 协议契约，以便直接通信。

While the [MCP protocol](https://modelcontextprotocol.io/docs/getting-started/intro) specification traditionally requires OAuth tokens for authentication, AgentCore runtime allows the ability to configure AWS IAM credentials for inbound requests to their MCP servers, addressing a crucial enterprise requirement.

虽然 [MCP 协议](https://modelcontextprotocol.io/docs/getting-started/intro)规范传统上要求使用 OAuth 令牌进行身份验证，但 AgentCore 运行时允许为其 MCP 服务器的入站请求配置 AWS IAM 凭证，以满足关键的企业需求。

### Tutorial Details

### 教程详情

| Information         | Details                                                   |
|:--------------------|:----------------------------------------------------------|
| Tutorial type       | Hosting Tools                                             |
| Tool type           | MCP server                                                |
| Tutorial components | Hosting MCP server on AgentCore Runtime                  |
| Tutorial vertical   | Cross-vertical                                            |
| Example complexity  | Easy                                                      |
| SDK used            | Amazon BedrockAgentCore Python SDK and MCP               |

| 信息                 | 详情                                                       |
|:--------------------|:----------------------------------------------------------|
| 教程类型             | 托管工具                                                   |
| 工具类型             | MCP 服务器                                                 |
| 教程组件             | 在 AgentCore Runtime 上托管 MCP 服务器                     |
| 教程行业             | 跨行业                                                     |
| 示例复杂度           | 简单                                                       |
| 使用的 SDK          | Amazon BedrockAgentCore Python SDK 和 MCP                 |

### Tutorial Architecture

### 教程架构

In this tutorial we will describe how to deploy an MCP server to AgentCore runtime.

在本教程中，我们将介绍如何将 MCP 服务器部署到 AgentCore 运行时。

For demonstration purposes, we will use a simple MCP server with 3 tools: `add_numbers`, `multiply_numbers` and `greet_user`

出于演示目的，我们将使用一个包含 3 个工具的简单 MCP 服务器：`add_numbers`（加法）、`multiply_numbers`（乘法）和 `greet_user`（用户问候）

<div style="text-align:left">
    <img src="images/hosting_mcp_server.png" width="60%"/>
</div>

### Tutorial Key Features

### 教程关键功能

* Creating MCP servers with custom tools
* Testing MCP servers locally
* Hosting MCP servers on Amazon Bedrock AgentCore Runtime
* Invoking deployed MCP servers with authentication

* 创建带有自定义工具的 MCP 服务器
* 本地测试 MCP 服务器
* 在 Amazon Bedrock AgentCore Runtime 上托管 MCP 服务器
* 使用身份验证调用已部署的 MCP 服务器

## Prerequisites

## 前提条件

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials configured
* Amazon Bedrock AgentCore SDK
* MCP (Model Context Protocol) library
* Running Docker daemon

要执行本教程，您需要：
* Python 3.10+
* 已配置的 AWS 凭证
* Amazon Bedrock AgentCore SDK
* MCP（模型上下文协议）库
* 运行中的 Docker 守护进程

In [1]:
%pip install -U -r requirements.txt

  Using cached bedrock_agentcore-0.1.5-py3-none-any.whl.metadata (7.0 kB)
  Using cached bedrock_agentcore_starter_toolkit-0.1.14-py3-none-any.whl.metadata (10.0 kB)
Using cached bedrock_agentcore_starter_toolkit-0.1.14-py3-none-any.whl (163 kB)
Using cached bedrock_agentcore-0.1.5-py3-none-any.whl (62 kB)
  Attempting uninstall: bedrock-agentcore
    Found existing installation: bedrock-agentcore 1.2.0
    Uninstalling bedrock-agentcore-1.2.0:
      Successfully uninstalled bedrock-agentcore-1.2.0
  Attempting uninstall: bedrock-agentcore-starter-toolkit
    Found existing installation: bedrock-agentcore-starter-toolkit 0.2.6
    Uninstalling bedrock-agentcore-starter-toolkit-0.2.6:
      Successfully uninstalled bedrock-agentcore-starter-toolkit-0.2.6
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os

In [3]:
boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client('ssm', region_name=region)

tool_name = "mcp_server_iam"

## Understanding MCP (Model Context Protocol)

## 理解 MCP（模型上下文协议）

MCP is a protocol that allows AI models to securely access external data and tools. Key concepts:

MCP 是一种允许 AI 模型安全访问外部数据和工具的协议。关键概念：

* **Tools**: Functions that the AI can call to perform actions
* **Streamable HTTP**: Transport protocol used by AgentCore Runtime
* **Session Isolation**: Each client gets isolated sessions via `Mcp-Session-Id` header
* **Stateless Operation**: Servers must support stateless operation for scalability

* **工具（Tools）**：AI 可以调用的用于执行操作的函数
* **可流式 HTTP（Streamable HTTP）**：AgentCore Runtime 使用的传输协议
* **会话隔离（Session Isolation）**：每个客户端通过 `Mcp-Session-Id` 头获得隔离的会话
* **无状态操作（Stateless Operation）**：服务器必须支持无状态操作以实现可扩展性

AgentCore Runtime expects MCP servers to be hosted on `0.0.0.0:8000/mcp` as the default path.

AgentCore Runtime 期望 MCP 服务器托管在 `0.0.0.0:8000/mcp` 作为默认路径。

### Project Structure

### 项目结构

Let's set up our project with the proper structure:

让我们用正确的结构设置我们的项目：

```
mcp_server_project/
├── mcp_server.py              # Main MCP server code / MCP 服务器主代码
├── mcp_client.py          # Local testing client / 本地测试客户端
├── mcp_client_remote.py   # Remote testing client / 远程测试客户端
├── requirements.txt          # Dependencies / 依赖项
└── __init__.py              # Python package marker / Python 包标记
```

## Creating MCP Server

## 创建 MCP 服务器

Now let's create our MCP server with three simple tools. The server uses FastMCP with `stateless_http=True` which is required for AgentCore Runtime compatibility.

现在让我们创建包含三个简单工具的 MCP 服务器。服务器使用 FastMCP 并设置 `stateless_http=True`，这是 AgentCore Runtime 兼容性所必需的。

In [4]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
from starlette.responses import JSONResponse

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b

@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers together"""
    return a * b

@mcp.tool()
def greet_user(name: str) -> str:
    """Greet a user by name"""
    return f"Hello, {name}! Nice to meet you."

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

Writing mcp_server.py


### What This Code Does

### 这段代码的作用

* **FastMCP**: Creates an MCP server that can host your tools
* **@mcp.tool()**: Decorator that turns your Python functions into MCP tools
* **stateless_http=True**: Required for AgentCore Runtime compatibility
* **Tools**: Three simple tools demonstrating different types of operations

* **FastMCP**：创建一个可以托管工具的 MCP 服务器
* **@mcp.tool()**：将 Python 函数转换为 MCP 工具的装饰器
* **stateless_http=True**：AgentCore Runtime 兼容性所必需的设置
* **工具**：三个简单的工具，演示不同类型的操作

## Creating Local Testing Client

## 创建本地测试客户端

Before deploying to AgentCore Runtime, let's create a client to test our MCP server locally:

在部署到 AgentCore Runtime 之前，让我们创建一个客户端来本地测试我们的 MCP 服务器：

In [5]:
%%writefile mcp_client.py
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

Writing mcp_client.py


 ### Testing Locally

 ### 本地测试

To test your MCP server locally:

要在本地测试您的 MCP 服务器：

1. **Terminal 1**: Start the MCP server
   ```bash
   python mcp_server.py
   ```

1. **终端 1**：启动 MCP 服务器
   ```bash
   python mcp_server.py
   ```
   
2. **Terminal 2**: Run the test client
   ```bash
   python mcp_client.py
   ```

2. **终端 2**：运行测试客户端
   ```bash
   python mcp_client.py
   ```

You should see your three tools listed in the output.

您应该在输出中看到列出的三个工具。

## Configuring AgentCore Runtime Deployment

## 配置 AgentCore Runtime 部署

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

接下来，我们将使用我们的启动工具包来配置 AgentCore Runtime 部署，包括入口点、我们刚创建的执行角色和需求文件。我们还将配置启动工具包在启动时自动创建 Amazon ECR 仓库。

During the configure step, your docker file will be generated based on your application code

在配置步骤中，将根据您的应用程序代码生成 docker 文件

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [6]:
print(f"Using AWS region: {region}")

required_files = ["mcp_server.py", "requirements.txt"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="MCP",
    agent_name=tool_name,
)
print("Configuration completed ✓")

Entrypoint parsed: file=D:\Documents\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\02-hosting-MCP-server\mcp_server.py, bedrock_agentcore_name=mcp_server
Configuring BedrockAgentCore agent: mcp_server_iam


Using AWS region: us-east-1
All required files found ✓
Configuring AgentCore Runtime...


⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
For deployment options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated .dockerignore
Generated Dockerfile: d:\Documents\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\02-hosting-MCP-server\Dockerfile
Generated .dockerignore: d:\Documents\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\02-hosting-MCP-server\.dockerignore
Setting 'mcp_server_iam' as default agent
Bedrock AgentCore configured: d:\Documents\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\02-hosting-MCP-server\.bedrock_agentcore.yaml


Configuration completed ✓


## Launching MCP Server to AgentCore Runtime

## 启动 MCP 服务器到 AgentCore Runtime

Now that we've got a docker file, let's launch the MCP server to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

现在我们已经有了 docker 文件，让我们将 MCP 服务器启动到 AgentCore Runtime。这将创建 Amazon ECR 仓库和 AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [7]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'mcp_server_iam' to account 710560201993 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: mcp_server_iam


Launching MCP server to AgentCore Runtime...
This may take several minutes...
Repository doesn't exist, creating new ECR repository: bedrock-agentcore-mcp_server_iam


✅ ECR repository available: 710560201993.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-mcp_server_iam
Getting or creating execution role for agent: mcp_server_iam
Using AWS region: us-east-1, account ID: 710560201993
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-22e9392f39
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-22e9392f39
Starting execution role creation process for agent: mcp_server_iam
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-east-1-22e9392f39
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-22e9392f39
✓ Role created: arn:aws:iam::710560201993:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-22e9392f39
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-mcp_server_iam
Role creation complete and ready for use with Bedrock AgentCore
✅ Execution role available: arn:aws:iam::710560201993:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-22e9392f39
Preparing CodeBuild project and uploadi

Launch completed ✓
Agent ARN: arn:aws:bedrock-agentcore:us-east-1:710560201993:runtime/mcp_server_iam-gzZxT7AVTh
Agent ID: mcp_server_iam-gzZxT7AVTh


In [8]:

agent_arn_response = ssm_client.put_parameter(
    Name='/mcp_server/runtime_iam/agent_arn',
    Value=launch_result.agent_arn,
    Type='String',
    Description='Agent ARN for MCP server with inbound auth',
    Overwrite=True
)
print("✓ Agent ARN stored in Parameter Store")

print("\nConfiguration stored successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")

✓ Agent ARN stored in Parameter Store

Configuration stored successfully!
Agent ARN: arn:aws:bedrock-agentcore:us-east-1:710560201993:runtime/mcp_server_iam-gzZxT7AVTh


## Creating Remote Testing Client

## 创建远程测试客户端

Now let's create a client to test our deployed MCP server. This client will retrieve the necessary credentials from AWS and connect to the deployed server:

现在让我们创建一个客户端来测试我们部署的 MCP 服务器。此客户端将从 AWS 检索必要的凭证并连接到已部署的服务器：

In [9]:
%%writefile mcp_client_remote.py       
import asyncio
import sys
import logging
import boto3
from boto3.session import Session
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from streamable_http_sigv4 import streamablehttp_client_with_sigv4


logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str
):
    """
    Create a streamable HTTP transport with AWS SigV4 authentication.

    This function creates an MCP client transport that uses AWS Signature Version 4 (SigV4)
    to authenticate requests. This is necessary because standard MCP clients don't natively
    support AWS IAM authentication, and this bridges that gap.

    Args:
        mcp_url (str): The URL of the MCP gateway endpoint
        service_name (str): The AWS service name for SigV4 signing (typically "bedrock-agentcore")
        region (str): The AWS region where the gateway is deployed

    Returns:
        StreamableHTTPTransportWithSigV4: A transport instance configured for SigV4 auth

    Example:
        >>> transport = create_streamable_http_transport_sigv4(
        ...     mcp_url=".../mcp",
        ...     service_name="bedrock-agentcore",
        ...     region="us-west-2"
        ... )
    """
    # Get AWS credentials from the current boto3 session
    # These credentials will be used to sign requests with SigV4
    session = boto3.Session()
    credentials = session.get_credentials()

    # Create and return the custom transport with SigV4 signing capability
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
    )


def get_full_tools_list(client):
    """
    Retrieve the complete list of tools from an MCP client, handling pagination.

    MCP servers may return tools in paginated responses. This function handles the
    pagination automatically and returns all available tools in a single list.

    Args:
        client: An MCP client instance (from strands.tools.mcp.mcp_client.MCPClient)

    Returns:
        list: A complete list of all tools available from the MCP server

    Example:
        >>> mcp_client = MCPClient(lambda: create_transport())
        >>> all_tools = get_full_tools_list(mcp_client)
        >>> print(f"Found {len(all_tools)} tools")
    """
    more_tools = True
    tools = []
    pagination_token = None

    # Loop until we've fetched all pages
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)

        tools.extend(tmp_tools)

        # Check if there are more pages to fetch
        if tmp_tools.pagination_token is None:
            # No more pages - we're done
            more_tools = False
        else:
            # More pages exist - prepare to fetch the next one
            more_tools = True
            pagination_token = tmp_tools.pagination_token

    return tools


async def main():
    boto_session = Session()
    region = boto_session.region_name
    print(f"Using AWS region: {region}")

    ssm_client = boto3.client("ssm", region_name=region)

    agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
    )
    agent_arn = agent_arn_response["Parameter"]["Value"]
    print(f"Retrieved Agent ARN: {agent_arn}")

    if not agent_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)

    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    try:
        async with create_streamable_http_transport_sigv4(
            mcp_url=mcp_url, service_name="bedrock-agentcore", region=region
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")

                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()

                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, "inputSchema") and tool.inputSchema:
                        properties = tool.inputSchema.get("properties", {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()

                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    asyncio.run(main())


Writing mcp_client_remote.py


## Testing Your Deployed MCP Server

## 测试您部署的 MCP 服务器

Let's test our deployed MCP server using the remote client:

让我们使用远程客户端测试已部署的 MCP 服务器：

In [10]:
print("Testing deployed MCP server...")
print("=" * 50)
!python mcp_client_remote.py

Testing deployed MCP server...
Using AWS region: us-east-1
Retrieved Agent ARN: arn:aws:bedrock-agentcore:us-east-1:710560201993:runtime/mcp_server_iam-gzZxT7AVTh

🔄 Initializing MCP session...
✓ MCP session initialized

🔄 Listing available tools...

📋 Available MCP Tools:
🔧 add_numbers
   Description: Add two numbers together
   Parameters: ['a', 'b']

🔧 multiply_numbers
   Description: Multiply two numbers together
   Parameters: ['a', 'b']

🔧 greet_user
   Description: Greet a user by name
   Parameters: ['name']

✅ Successfully connected to MCP server!
Found 3 tools available.


2026-01-20 13:46:30,762 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-01-20 13:46:31,961 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-01-20 13:46:33,150 - httpx - INFO - HTTP Request: POST https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A710560201993%3Aruntime%2Fmcp_server_iam-gzZxT7AVTh/invocations?qualifier=DEFAULT "HTTP/1.1 200 OK"
2026-01-20 13:46:33,150 - mcp.client.streamable_http - INFO - Received session ID: e72e4b07-f1d2-48a4-a9d7-2630dd2f06b6
2026-01-20 13:46:33,150 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25
2026-01-20 13:46:33,984 - httpx - INFO - HTTP Request: GET https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A710560201993%3Aruntime%2Fmcp_server_iam-gzZxT7AVTh/invocations?qualifier=DEFAULT "HTTP/1.1 404 Not Found"
2026-0

### Invoking MCP Tools Remotely

### 远程调用 MCP 工具

Now let's create an enhanced client that not only lists tools but also invokes them to demonstrate the full MCP functionality:

现在让我们创建一个增强版客户端，它不仅可以列出工具，还可以调用它们来演示完整的 MCP 功能：

In [11]:
%%writefile invoke_mcp_tools.py
import asyncio
import sys
import os
import logging
import boto3
from boto3.session import Session
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from streamable_http_sigv4 import streamablehttp_client_with_sigv4

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str
):
    """
    Create a streamable HTTP transport with AWS SigV4 authentication.

    This function creates an MCP client transport that uses AWS Signature Version 4 (SigV4)
    to authenticate requests. This is necessary because standard MCP clients don't natively
    support AWS IAM authentication, and this bridges that gap.

    Args:
        mcp_url (str): The URL of the MCP gateway endpoint
        service_name (str): The AWS service name for SigV4 signing (typically "bedrock-agentcore")
        region (str): The AWS region where the gateway is deployed

    Returns:
        StreamableHTTPTransportWithSigV4: A transport instance configured for SigV4 auth

    Example:
        >>> transport = create_streamable_http_transport_sigv4(
        ...     mcp_url=".../mcp",
        ...     service_name="bedrock-agentcore",
        ...     region="us-west-2"
        ... )
    """
    # Get AWS credentials from the current boto3 session
    # These credentials will be used to sign requests with SigV4
    session = boto3.Session()
    credentials = session.get_credentials()

    # Create and return the custom transport with SigV4 signing capability
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
    )


def get_full_tools_list(client):
    """
    Retrieve the complete list of tools from an MCP client, handling pagination.

    MCP servers may return tools in paginated responses. This function handles the
    pagination automatically and returns all available tools in a single list.

    Args:
        client: An MCP client instance (from strands.tools.mcp.mcp_client.MCPClient)

    Returns:
        list: A complete list of all tools available from the MCP server

    Example:
        >>> mcp_client = MCPClient(lambda: create_transport())
        >>> all_tools = get_full_tools_list(mcp_client)
        >>> print(f"Found {len(all_tools)} tools")
    """
    more_tools = True
    tools = []
    pagination_token = None

    # Loop until we've fetched all pages
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)

        tools.extend(tmp_tools)

        # Check if there are more pages to fetch
        if tmp_tools.pagination_token is None:
            # No more pages - we're done
            more_tools = False
        else:
            # More pages exist - prepare to fetch the next one
            more_tools = True
            pagination_token = tmp_tools.pagination_token

    return tools


async def main():
    boto_session = Session()
    region = boto_session.region_name
    print(f"Using AWS region: {region}")

    ssm_client = boto3.client("ssm", region_name=region)

    agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
    )
    agent_arn = agent_arn_response["Parameter"]["Value"]
    print(f"Retrieved Agent ARN: {agent_arn}")

    if not agent_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)

    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    try:
        async with create_streamable_http_transport_sigv4(
                mcp_url=mcp_url, service_name="bedrock-agentcore", region=region
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")

                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()

                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")

                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)

                try:
                    print("\n➕ Testing add_numbers(5, 3)...")
                    add_result = await session.call_tool(
                        name="add_numbers", arguments={"a": 5, "b": 3}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n✖️  Testing multiply_numbers(4, 7)...")
                    multiply_result = await session.call_tool(
                        name="multiply_numbers", arguments={"a": 4, "b": 7}
                    )
                    print(f"   Result: {multiply_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n👋 Testing greet_user('Alice')...")
                    greet_result = await session.call_tool(
                        name="greet_user", arguments={"name": "Alice"}
                    )
                    print(f"   Result: {greet_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                print("\n✅ MCP tool testing completed!")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    asyncio.run(main())


Writing invoke_mcp_tools.py


## Test Tool Invocation

## 测试工具调用

Let's test our MCP tools by actually invoking them:

让我们通过实际调用来测试我们的 MCP 工具：

In [12]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

Testing MCP tool invocation...
Using AWS region: us-east-1
Retrieved Agent ARN: arn:aws:bedrock-agentcore:us-east-1:710560201993:runtime/mcp_server_iam-gzZxT7AVTh

🔄 Initializing MCP session...
✓ MCP session initialized

🔄 Listing available tools...

📋 Available MCP Tools:
🔧 add_numbers: Add two numbers together
🔧 multiply_numbers: Multiply two numbers together
🔧 greet_user: Greet a user by name

🧪 Testing MCP Tools:

➕ Testing add_numbers(5, 3)...
   Result: 8

✖️  Testing multiply_numbers(4, 7)...
   Result: 28

👋 Testing greet_user('Alice')...
   Result: Hello, Alice! Nice to meet you.

✅ MCP tool testing completed!


2026-01-20 13:47:07,230 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-01-20 13:47:08,389 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials
2026-01-20 13:47:09,689 - httpx - INFO - HTTP Request: POST https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A710560201993%3Aruntime%2Fmcp_server_iam-gzZxT7AVTh/invocations?qualifier=DEFAULT "HTTP/1.1 200 OK"
2026-01-20 13:47:09,689 - mcp.client.streamable_http - INFO - Received session ID: a7f109dc-e453-46c5-bf3d-33b0e004f707
2026-01-20 13:47:09,689 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25
2026-01-20 13:47:10,535 - httpx - INFO - HTTP Request: GET https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A710560201993%3Aruntime%2Fmcp_server_iam-gzZxT7AVTh/invocations?qualifier=DEFAULT "HTTP/1.1 404 Not Found"
2026-0

## Next Steps

## 后续步骤

Now that you have successfully deployed an MCP server to AgentCore Runtime, you can:

现在您已成功将 MCP 服务器部署到 AgentCore Runtime，您可以：

1. **Add More Tools**: Extend your MCP server with additional tools
2. **Custom Authentication**: Implement AWS IAM inbound authentication
3. **Integration**: Integrate with other AgentCore services

1. **添加更多工具**：使用额外的工具扩展您的 MCP 服务器
2. **自定义身份验证**：实现 AWS IAM 入站身份验证
3. **集成**：与其他 AgentCore 服务集成

## Cleanup (Optional)

## 清理（可选）

If you want to clean up the resources created during this tutorial, run the following cells:

如果您想清理本教程中创建的资源，请运行以下单元格：

In [13]:
try:
    ssm_client.delete_parameter(Name='/mcp_server/runtime_iam/agent_arn')
    print("✓ Parameter Store parameter deleted")
except ssm_client.exceptions.ParameterNotFound:
    print("ℹ️  Parameter Store parameter not found")

✓ Parameter Store parameter deleted


In [14]:
destroy_bedrock_agentcore(
    config_path=Path(".bedrock_agentcore.yaml"),
    agent_name=tool_name,
    delete_ecr_repo=True
)

Starting destroy operation for agent: mcp_server_iam (dry_run=False, delete_ecr_repo=True)
Skipping deletion of DEFAULT endpoint
Deleted AgentCore agent: arn:aws:bedrock-agentcore:us-east-1:710560201993:runtime/mcp_server_iam-gzZxT7AVTh
Checking ECR repository: bedrock-agentcore-mcp_server_iam in region: us-east-1
Deleted 1 ECR images from bedrock-agentcore-mcp_server_iam
Deleted ECR repository: bedrock-agentcore-mcp_server_iam
Deleted CodeBuild project: bedrock-agentcore-mcp_server_iam-builder
Deleted inline policy CodeBuildExecutionPolicy from role AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-22e9392f39
Deleted CodeBuild IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-22e9392f39
Deleted IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-22e9392f39
Removed agent configuration: mcp_server_iam
Cleared default agent (no agents remaining)
Removed configuration file: .bedrock_agentcore.yaml
Destroy operation completed. Resources removed: 8, Warnings: 1, Errors: 0


DestroyResult(agent_name='mcp_server_iam', resources_removed=['AgentCore agent: arn:aws:bedrock-agentcore:us-east-1:710560201993:runtime/mcp_server_iam-gzZxT7AVTh', 'ECR images: 1 images from bedrock-agentcore-mcp_server_iam', 'ECR repository: bedrock-agentcore-mcp_server_iam', 'CodeBuild project: bedrock-agentcore-mcp_server_iam-builder', 'Deleted CodeBuild IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-22e9392f39', 'IAM execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-22e9392f39', 'Agent configuration: mcp_server_iam', 'Configuration file (no agents remaining)'], warnings=['DEFAULT endpoint cannot be explicitly deleted, skipping'], errors=[], dry_run=False)

# 🎉 Congratulations!

# 🎉 恭喜！

You have successfully:

您已成功完成：

✅ **Created an MCP server** with custom tools  
✅ **Tested locally** using MCP client  
✅ **Set up authentication** with Amazon Cognito  
✅ **Deployed to AWS** using AgentCore Runtime  
✅ **Invoked remotely** with proper authentication  
✅ **Learned MCP concepts** and best practices  

✅ **创建了 MCP 服务器**，包含自定义工具  
✅ **本地测试**，使用 MCP 客户端  
✅ **设置了身份验证**，使用 Amazon Cognito  
✅ **部署到 AWS**，使用 AgentCore Runtime  
✅ **远程调用**，使用正确的身份验证  
✅ **学习了 MCP 概念**和最佳实践  

Your MCP server is now running on Amazon Bedrock AgentCore Runtime and ready for production use!

您的 MCP 服务器现在正在 Amazon Bedrock AgentCore Runtime 上运行，并已准备好用于生产环境！

## Summary

## 总结

In this tutorial, you learned how to:
- Build MCP servers using FastMCP
- Configure stateless HTTP transport for AgentCore compatibility
- Set up AWS IAM inbound authentication
- Deploy and manage MCP servers on AWS
- Test both locally and remotely
- Use MCP clients for tool invocation

在本教程中，您学习了如何：
- 使用 FastMCP 构建 MCP 服务器
- 为 AgentCore 兼容性配置无状态 HTTP 传输
- 设置 AWS IAM 入站身份验证
- 在 AWS 上部署和管理 MCP 服务器
- 本地和远程测试
- 使用 MCP 客户端进行工具调用

The deployed MCP server can now be integrated into larger AI applications and workflows!

部署的 MCP 服务器现在可以集成到更大的 AI 应用程序和工作流程中！